# VFCFinder results vs. ground truth

Compares the top-ranked commit from each `ranked_<GHSA>[_<fixed_version>].csv` (row 1 = predicted VFC) with `VFCCommitSHA` in `tests/data/sample/ghsa_annotation_sample.json`.
Results are read from `output/results/first_20_samples` and `output/results/second_20_samples`, which hold the results for the first and last 20 entries of the annotation file.
Advisories with an empty `VFCCommitSHA` have no ground truth, so they are listed but not scored.

In [6]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# find the repo root from wherever the notebook is opened
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tests/data/sample").exists()
)
ANNOTATION_PATH = REPO_ROOT / "tests/data/sample/ghsa_annotation_sample.json"
RESULTS_ROOT = REPO_ROOT / "output/results"

# result folders, in annotation-file order: entries 1-20, then entries 21-40
SAMPLES = ["first_20_samples", "second_20_samples"]
SAMPLE_SIZE = 20

## Ground truth

In [7]:
annotations = pd.read_json(ANNOTATION_PATH, dtype=str)
annotations["VFCCommitSHA"] = annotations["VFCCommitSHA"].fillna("").str.strip().str.lower()
annotations["has_ground_truth"] = annotations["VFCCommitSHA"] != ""

# a GHSA can be listed for several fixed versions, so the sample is part of the key
annotations["sample"] = [SAMPLES[i // SAMPLE_SIZE] for i in range(len(annotations))]
annotations[["sample", "ghsa_id", "fixed_version", "has_ground_truth", "VFCCommitSHA"]]

,sample,ghsa_id,fixed_version,has_ground_truth,VFCCommitSHA
0,first_20_samples,GHSA-xph3-r2jf-4vp3,3.1.4,False,
1,first_20_samples,GHSA-4w46-w44m-3jq3,4.5.0,True,da905a357d062ab4fea727a21eac231acc2ed92a
2,first_20_samples,GHSA-2phv-j68v-wwqx,10.27.0,False,
3,first_20_samples,GHSA-9pp3-53p2-ww9v,3.6.2,False,
4,first_20_samples,GHSA-99h5-pjcv-gr6v,1.3.26,True,556085067609c508f8c546ceef9003ee8c607d39
5,first_20_samples,GHSA-8xwg-wv7v-4vqp,1.8.4,False,
6,first_20_samples,GHSA-5xf4-f2fq-f69j,1.22.0,True,039bafd74b7b1a88a53a54f8fa6fa872615e90e7
7,first_20_samples,GHSA-4vq7-882g-wcg4,5.23.0,False,
8,first_20_samples,GHSA-v273-448j-v4qj,10.25.5,True,f41c1fc02fe901598f3328118b42b13bc6bc9b04
9,first_20_samples,GHSA-49x3-8228-3w3m,3.12.0,True,ca617df135e78833f93c8320cb2d2cf8bba809f5


## VFCFinder predictions

The GHSA ID and fixed version come from the file name (`ranked_<GHSA>_<fixed_version>.csv`). Older result files named `ranked_<GHSA>.csv` get their fixed version from the annotation entry of the same sample. Rows are sorted by `ranking_prob` (descending) so the first row is the top-ranked commit.

In [8]:
rows = []
ranked = {}  # (sample, ghsa_id, fixed_version) -> full ranked dataframe, used for the rank lookup below
for sample in SAMPLES:
    for csv_path in sorted((RESULTS_ROOT / sample).glob("ranked_*.csv")):
        ghsa_id, _, fixed_version = csv_path.stem.removeprefix("ranked_").partition("_")
        if not fixed_version:
            # old file name without a version, use the sample's annotation entry
            versions = annotations.loc[
                (annotations["sample"] == sample) & (annotations["ghsa_id"] == ghsa_id),
                "fixed_version",
            ].tolist()
            if len(versions) != 1:
                raise ValueError(f"{csv_path.name}: cannot tell the fixed version, annotation has {versions}")
            fixed_version = versions[0]

        df = pd.read_csv(csv_path).sort_values("ranking_prob", ascending=False, kind="stable")
        df = df.reset_index(drop=True)
        ranked[(sample, ghsa_id, fixed_version)] = df
        top = df.iloc[0]
        rows.append(
            {
                "sample": sample,
                "ghsa_id": ghsa_id,
                "fixed_version": fixed_version,
                "predicted_sha": top["sha"].lower(),
                "ranking_prob": top["ranking_prob"],
                "candidate_commits": len(df),
                "predicted_message": top["message"],
            }
        )

predictions = pd.DataFrame(rows)
predictions[["sample", "ghsa_id", "fixed_version", "predicted_sha", "ranking_prob", "candidate_commits"]]

,sample,ghsa_id,fixed_version,predicted_sha,ranking_prob,candidate_commits
0,first_20_samples,GHSA-2phv-j68v-wwqx,10.27.0,5d983ba175a658dae65597dda0f4de04fdc54948,0.088440,15
1,first_20_samples,GHSA-2q7r-29rg-6m5h,12.5.0,4d9795cd5b57a36756d37b7f036eae369f69fa66,0.107930,3
2,first_20_samples,GHSA-2qgm-m29m-cj2h,1.23.16,6cfae01a0d3727c517afe512fc8fec1d99acf875,0.160546,3
3,first_20_samples,GHSA-49x3-8228-3w3m,3.12.0,1d4ffda1f8a74f9e6804cc843404da3b7e1806fd,0.031484,117
4,first_20_samples,GHSA-4hjh-wcwx-xvwj,0.30.2,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,0.060061,4
5,first_20_samples,GHSA-4vq7-882g-wcg4,5.23.0,01adb034f24727d3bb321bbbb6696a7f4cd91689,0.107442,47
6,first_20_samples,GHSA-4w46-w44m-3jq3,4.5.0,da905a357d062ab4fea727a21eac231acc2ed92a,0.130528,64
7,first_20_samples,GHSA-5xf4-f2fq-f69j,1.22.0,1b334e687baa75c84440def211cca0636879d14d,0.056844,18
8,first_20_samples,GHSA-8vch-m3f4-q8jf,1.4.17,3af978663e437dccc6c1a2a3aff4b74e1574849e,0.135235,62
9,first_20_samples,GHSA-8xwg-wv7v-4vqp,1.8.4,fe2dfdc24d876f4233ab0f27e9759c6956ac5235,0.063388,25


## Comparison - we ignore no ground-truth samples in the comparison

In [9]:
def rank_of(sample, ghsa_id, fixed_version, sha):
    """1-based position of sha in the ranked CSV, None if it is not a candidate"""
    key = (sample, ghsa_id, fixed_version)
    if key not in ranked or not sha:
        return None
    matches = ranked[key].index[ranked[key]["sha"].str.lower() == sha]
    return int(matches[0]) + 1 if len(matches) else None


# only advisories with a ground-truth VFC are compared
comparison = (
    annotations[annotations["has_ground_truth"]]
    .merge(predictions, on=["sample", "ghsa_id", "fixed_version"], how="left")
    .reset_index(drop=True)
)
comparison["ground_truth_rank"] = [
    rank_of(r.sample, r.ghsa_id, r.fixed_version, r.VFCCommitSHA)
    for r in comparison.itertuples()
]


def status(r):
    if pd.isna(r["predicted_sha"]):
        return "no result CSV"
    return "match" if r["predicted_sha"] == r["VFCCommitSHA"] else "mismatch"


comparison["status"] = comparison.apply(status, axis=1)

comparison[
    ["sample", "ghsa_id", "fixed_version", "status", "VFCCommitSHA", "predicted_sha", "ground_truth_rank", "candidate_commits"]
].rename(columns={"VFCCommitSHA": "ground_truth_sha"})

,sample,ghsa_id,fixed_version,status,ground_truth_sha,predicted_sha,ground_truth_rank,candidate_commits
0,first_20_samples,GHSA-4w46-w44m-3jq3,4.5.0,match,da905a357d062ab4fea727a21eac231acc2ed92a,da905a357d062ab4fea727a21eac231acc2ed92a,1,64
1,first_20_samples,GHSA-99h5-pjcv-gr6v,1.3.26,match,556085067609c508f8c546ceef9003ee8c607d39,556085067609c508f8c546ceef9003ee8c607d39,1,6
2,first_20_samples,GHSA-5xf4-f2fq-f69j,1.22.0,mismatch,039bafd74b7b1a88a53a54f8fa6fa872615e90e7,1b334e687baa75c84440def211cca0636879d14d,11,18
3,first_20_samples,GHSA-v273-448j-v4qj,10.25.5,match,f41c1fc02fe901598f3328118b42b13bc6bc9b04,f41c1fc02fe901598f3328118b42b13bc6bc9b04,1,6
4,first_20_samples,GHSA-49x3-8228-3w3m,3.12.0,mismatch,ca617df135e78833f93c8320cb2d2cf8bba809f5,1d4ffda1f8a74f9e6804cc843404da3b7e1806fd,2,117
5,first_20_samples,GHSA-w87r-vg9q-crqm,8.8.5,match,9ef6d3c9962c4ba01e3fb8075855570c192b4681,9ef6d3c9962c4ba01e3fb8075855570c192b4681,1,14
6,first_20_samples,GHSA-2qgm-m29m-cj2h,1.23.16,match,6cfae01a0d3727c517afe512fc8fec1d99acf875,6cfae01a0d3727c517afe512fc8fec1d99acf875,1,3
7,first_20_samples,GHSA-2q7r-29rg-6m5h,12.5.0,match,4d9795cd5b57a36756d37b7f036eae369f69fa66,4d9795cd5b57a36756d37b7f036eae369f69fa66,1,3
8,first_20_samples,GHSA-4hjh-wcwx-xvwj,0.30.2,match,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,1,4
9,first_20_samples,GHSA-8vch-m3f4-q8jf,1.4.17,mismatch,26935bf76ebc43b4a43d48b173fc853de43bb51e,3af978663e437dccc6c1a2a3aff4b74e1574849e,2,62


## Summary per sample

In [10]:
summary = (
    comparison.groupby(["sample", "status"]).size().unstack(fill_value=0)
    .reindex(columns=["match", "mismatch", "no result CSV"], fill_value=0)
)
summary["scored"] = summary["match"] + summary["mismatch"]
summary["top1_accuracy"] = (summary["match"] / summary["scored"]).round(3)
summary.loc["all"] = summary.sum(numeric_only=True)
summary.loc["all", "top1_accuracy"] = round(summary.loc["all", "match"] / summary.loc["all", "scored"], 3)
count_cols = ["match", "mismatch", "no result CSV", "scored"]
summary[count_cols] = summary[count_cols].astype(int)
summary

status,match,mismatch,no result CSV,scored,top1_accuracy
sample,,,,,
first_20_samples,9,4,0,13,0.692
second_20_samples,7,5,0,12,0.583
all,16,9,0,25,0.640
